# Import data from AEMO

In [0]:
# Systematically download aemo data
import os 
import requests
import time

OUT_DIR = "/Volumes/workspace/endava/aemo"
os.makedirs(OUT_DIR, exist_ok = True)
FILE_EXT = ".csv"

URL_TEMPLATE = "https://www.aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_{year}{month:02d}_{state}1.csv"

STATE = "QLD"
START_YEAR = 2006
END_YEAR = 2025

YEARS = range(START_YEAR, END_YEAR + 1)
MONTHS = range(1, 13)

for year in YEARS:
    for month in MONTHS:
        url = URL_TEMPLATE.format(state = STATE, year = year, month = month)
        filename = f"{OUT_DIR}/{STATE}-{year}-{month:02d}{FILE_EXT}"
        if os.path.exists(filename):
            print(f"Skipping {filename} (already exists)")
            continue

        try:
            print(f"Downloading {url} to {filename}")
            response = requests.get(url, timeout = 60)
            if response.status_code == 404:
                print(f"Not found: {url}")
                continue
            response.raise_for_status()
            with open(filename, "wb") as f:
                f.write(response.content)
            time.sleep(0.3)
        except Exception as e:
            print(f"Failed {url}: {e}")

print(f"\nAll files saved to: {OUT_DIR}")

In [0]:
# Investigate the first subset of files
import glob, os
import pandas as pd

DATA_DIR = "/Volumes/workspace/endava/aemo"
files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
files.sort()

# Test out with a manageable subset first including last 12 files
sample = pd.concat(
    (pd.read_csv(f) for f in files[-12:]),
    ignore_index = True
)

# AEMO columns often include: REGION, SETTLEMENTDATE, TOTALDEMAND, RRP, PERIODTYPE
sample["SETTLEMENTDATE"] = pd.to_datetime(sample["SETTLEMENTDATE"], errors = "coerce")
for col in ["TOTALDEMAND", "RRP"]:
    sample[col] = pd.to_numeric(sample[col], errors = "coerce")
    
for col in ["REGION", "PERIODTYPE"]:
    sample[col] = sample[col].astype("category")

print(sample.head())
print(sample.dtypes)
print(sample.describe(include = "all"))

In [0]:
# Try it for all files
df = pd.concat((pd.read_csv(f) for f in files), ignore_index = True)

# Cleaning
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"], errors = "coerce")
for col in ["TOTALDEMAND", "RRP"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce")
for col in ["REGION", "PERIODTYPE"]:
    df[col] = df[col].astype("category")

In [0]:
df